In [ ]:
from sagemaker.workflow.function_step import step
from sagemaker.workflow.pipeline import Pipeline
import sagemaker
from sagemaker.workflow.parameters import ParameterInteger
from sagemaker.workflow.execution_variables import ExecutionVariables
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.workflow.fail_step import FailStep
import boto3

In [ ]:
sts = boto3.client("sts")
account_id = sts.get_caller_identity()["Account"]

region = boto3.Session().region_name
print(f"Region: {region}")
user = "grupo5"

TRACKING_SERVER = "mlflow-tracking-server-grupo5"
default_bucket = "s3-mlflow-artifacts-mlflow-tracking-server-grupo5-01"
default_prefix = f"sagemaker/clients-attrition/{user}"
default_path = default_bucket + "/" + default_prefix
athena_database = "glue-catalog-database-utec-bank-01"

role = sagemaker.get_execution_role()

In [ ]:
sagemaker_session = sagemaker.Session(default_bucket=default_bucket,
                                      default_bucket_prefix=default_prefix)
# #Pipeline configuration
instance_type = "ml.m5.large"
pipeline_name = f"pipeline-train-{user}"
model_name = f"credit-card-fraud-detection-{user}"

# MLFlow configuration
tracking_server_arn = f"arn:aws:sagemaker:{region}:{account_id}:mlflow-tracking-server/{TRACKING_SERVER}"
experiment_name = f"pipeline-train-exp-{user}"

In [ ]:
edad_init = ParameterInteger(name="EdadInit")
edad_end = ParameterInteger(name="EdadEnd")

In [ ]:
# jalar datos para el entrenamiento
@step(
    name="DataPull",
    instance_type=instance_type,
    dependencies="./data_pull_requirements.txt"
)
def data_pull(experiment_name: str, run_name: str,
              edad_init: int, edad_end: int) -> tuple[str, str, str]:
  import awswrangler as wr
  import mlflow

  mlflow.set_tracking_uri(tracking_server_arn)
  mlflow.set_experiment(experiment_name)
  TARGET_COL = "attrition"
  query = """
        SELECT  id_correlativo,
                codmes,
                flg_bancarizado,
                rang_ingreso, 
                flag_lima_provincia, 
                edad, 
                antiguedad, 
                attrition,
                rang_sdo_pasivo_menos0, 
                sdo_activo_menos0,
                sdo_activo_menos1,
                sdo_activo_menos2,
                sdo_activo_menos3,
                sdo_activo_menos4,
                sdo_activo_menos5,
                flg_seguro_menos0,
                flg_seguro_menos1,
                flg_seguro_menos2,
                flg_seguro_menos3,
                flg_seguro_menos4,
                flg_seguro_menos5,
                rang_nro_productos_menos0, 
                flg_nomina,
                nro_acces_canal1_menos0,
                nro_acces_canal1_menos1,
                nro_acces_canal1_menos2,
                nro_acces_canal1_menos3,
                nro_acces_canal1_menos4,
                nro_acces_canal1_menos5,
                nro_acces_canal2_menos0,
                nro_acces_canal2_menos1,
                nro_acces_canal2_menos2,
                nro_acces_canal2_menos3,
                nro_acces_canal2_menos4,
                nro_acces_canal2_menos5,
                nro_acces_canal3_menos0,
                nro_acces_canal3_menos1,
                nro_acces_canal3_menos2,
                nro_acces_canal3_menos3,
                nro_acces_canal3_menos4,
                nro_acces_canal3_menos5,
                nro_entid_ssff_menos0,
                nro_entid_ssff_menos1,
                nro_entid_ssff_menos2,
                nro_entid_ssff_menos3,
                nro_entid_ssff_menos4,
                nro_entid_ssff_menos5,
                flg_sdo_otssff_menos0,
                flg_sdo_otssff_menos1,
                flg_sdo_otssff_menos2,
                flg_sdo_otssff_menos3,
                flg_sdo_otssff_menos4,
                flg_sdo_otssff_menos5
        FROM    "glue-catalog-database-utec-bank-01"."data_origen"
        WHERE   edad between {} and {}
    """.format(edad_init, edad_end)
  train_s3_path = f"s3://{default_path}/data_train/trained_clientes_pipeline.csv"
  with mlflow.start_run(run_name=run_name) as run:
    run_id = run.info.run_id
    with mlflow.start_run(run_name="DataPull", nested=True):
      df = wr.athena.read_sql_query(sql=query, database=athena_database)
      df.to_csv(train_s3_path, index=False)
      mlflow.log_input(
          mlflow.data.from_pandas(df, train_s3_path,
                                  targets=TARGET_COL),
          context="DataPull"
      )
  return train_s3_path, experiment_name, run_id

In [ ]:
# entrenamiento del modelo

@step(
    name="AttritionModelTraining",
    instance_type=instance_type,
    dependencies="./model_training_requirements.txt"
)
def model_training(train_s3_path: str, experiment_name: str,
                   run_id: str) -> tuple[str, str, str, str]:
  import pandas as pd
  import mlflow
  from sklearn.model_selection import train_test_split
  from sklearn.preprocessing import LabelEncoder
  from sklearn.impute import SimpleImputer
  from xgboost import XGBClassifier
  import numpy as np

  # Configuración
  TARGET_COL = "attrition"
  SEED = 42
  TRAIN_SPLIT = 0.7

  # Features seleccionadas para el modelo
  NUMERIC_FEATURES = [
        'flg_bancarizado', 'edad', 'antiguedad', 
        'sdo_activo_menos0', 'sdo_activo_menos1', 'sdo_activo_menos2',
        'flg_seguro_menos0', 'flg_seguro_menos1', 'flg_seguro_menos2',
        'flg_nomina',
        'nro_acces_canal1_menos0', 'nro_acces_canal2_menos0', 'nro_acces_canal3_menos0'
  ]
    
  # Configurar MLflow
  mlflow.set_tracking_uri(tracking_server_arn)
  mlflow.set_experiment(experiment_name)
    
  # Cargar y preparar datos
  df = pd.read_csv(train_s3_path)
  
  # Preprocesamiento
  df_processed = df.copy()
  
  # Codificar variables categóricas
  categorical_cols = ['flag_lima_provincia', 'rang_ingreso', 'rang_sdo_pasivo_menos0']
  label_encoders = {}
  
  for col in categorical_cols:
      le = LabelEncoder()
      df_processed[col] = df_processed[col].astype(str)
      df_processed[f'{col}_encoded'] = le.fit_transform(df_processed[col])
      label_encoders[col] = le
    
  # Imputar valores faltantes
  imputer = SimpleImputer(strategy='median')
  numeric_cols = ['edad', 'antiguedad']
  for col in numeric_cols:
      df_processed[col] = imputer.fit_transform(df_processed[[col]]).flatten()
  
  # Seleccionar features y target
  X = df_processed[FEATURES]
  y = df_processed[TARGET_COL]
  
  # División train/test
  X_train, X_test, y_train, y_test = train_test_split(
      X, y,
      train_size=TRAIN_SPLIT,
      random_state=SEED,
      stratify=y
  )
    
  # Parámetros del modelo XGBoost optimizados para attrition
  use_gpu = False
  param = dict(
      objective="binary:logistic",
      max_depth=4,  # Reducido para evitar overfitting con pocos datos
      eta=0.1,      # Learning rate más conservador
      gamma=1,
      min_child_weight=3,
      subsample=0.8,
      colsample_bytree=0.8,
      tree_method="gpu_hist" if use_gpu else "hist",
      n_estimators=100,
      scale_pos_weight=len(y_train[y_train==0]) / len(y_train[y_train==1]),  # Para desbalance
      eval_metric=['auc', 'logloss']
  )
    
  # Entrenamiento con MLflow
  with mlflow.start_run(run_id=run_id):
      with mlflow.start_run(run_name="AttritionModelTraining",
                            nested=True) as training_run:
          training_run_id = training_run.info.run_id
          
          # Guardar datos de test
          test_s3_path = f"s3://{default_path}/test_data/attrition_test.csv"
          df_test = pd.concat([X_test, y_test], axis=1)
          df_test.to_csv(test_s3_path, index=False)
          
          # Log del dataset de test
          mlflow.log_input(
              mlflow.data.from_pandas(df_test, test_s3_path,
                                      targets=TARGET_COL),
              context="AttritionModelTraining"
          )
          
          # Log métricas del dataset
          mlflow.log_metric("train_size", len(X_train))
          mlflow.log_metric("test_size", len(X_test))
          mlflow.log_metric("attrition_rate", y.mean())
          mlflow.log_metric("class_imbalance_ratio", 
                            len(y[y==0]) / len(y[y==1]))
          
          # Configurar autolog de XGBoost
          mlflow.xgboost.autolog(
              log_input_examples=True,
              log_model_signatures=True,
              log_models=True,
              log_datasets=True,
              model_format="xgb",
          )
          
          # Entrenar modelo
          xgb = XGBClassifier(**param)
          xgb.fit(
              X_train, y_train,
              eval_set=[(X_test, y_test)],
              verbose=False
          )
          
          # Log métricas adicionales
          from sklearn.metrics import f1_score, roc_auc_score, classification_report
          
          y_pred = xgb.predict(X_test)
          y_pred_proba = xgb.predict_proba(X_test)[:, 1]
          
          f1 = f1_score(y_test, y_pred)
          auc = roc_auc_score(y_test, y_pred_proba)
          
          mlflow.log_metric("f1_score", f1)
          mlflow.log_metric("auc_score", auc)
          
          # Log feature importance
          feature_importance = pd.DataFrame({
              'feature': FEATURES,
              'importance': xgb.feature_importances_
          }).sort_values('importance', ascending=False)
          
          mlflow.log_text(feature_importance.to_string(), "feature_importance.txt")

      # # Log preprocessing artifacts
      # mlflow.log_dict(
      #     {col: list(le.classes_) for col, le in label_encoders.items()},
      #     "label_encoders.json"
      # )

  return test_s3_path, experiment_name, run_id, training_run_id

In [ ]:
data_pull_step = data_pull(experiment_name=experiment_name,
                           run_name=ExecutionVariables.PIPELINE_EXECUTION_ID,
                           edad_init=edad_init,
                           edad_end=edad_end)

model_training_step = model_training(train_s3_path=data_pull_step[0],
                                     experiment_name=data_pull_step[1],
                                     run_id=data_pull_step[2])
# agregar mas steps

In [ ]:
steps = [
    data_pull_step,
    model_training_step,
    # conditional_register_step
]
pipeline = Pipeline(name=pipeline_name,
                    steps=steps,
                    parameters=[edad_init, edad_end])
pipeline.upsert(role_arn=role)

In [ ]:
import random
import string
from datetime import datetime


def generate_random_string(length=8):
  return ''.join(random.choices(string.ascii_lowercase + string.digits, k=length))


timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
random_suffix = generate_random_string(4)

pipeline.start(parameters={"EdadInit": 27,
                           "EdadEnd": 28},
               execution_display_name=f"exec-{timestamp}-{random_suffix}",
               execution_description=f"Pipeline run at {timestamp}")